# From pandas to Spark: the five years of MediaMas 🐼➡️⚡

In Session 3 you analysed **BlacFraidi** with Spark. The management of **MediaMas** was delighted and now asks for more: *the same analysis for **all the campaigns of the last years**, and next year again, with more stores and more data*.

Your tool of choice is **pandas**, the favourite library of every data analyst. Today you will find out **how far pandas goes**, why it stops, and how its code **scales up to Spark** without rewriting everything: with the **PySpark DataFrame API**, with the **pandas API on Spark** (`pyspark.pandas`) and with **pandas UDFs** (your pandas code running *inside* Spark).

| Part | What you will do |
|---|---|
| 0 | 🔁 Install Java + PySpark 4.3.0 and start a `SparkSession` (ready-made cells) |
| 1 | 🐼 **pandas**: the analysis of `TrainBF` (250 000 rows) with vectorised code, `groupby`, `pivot_table` and `merge` |
| 2 | 🧱 **The wall**: how much memory pandas needs and how its time grows when the data grows |
| 3 | ⚡ **PySpark DataFrames**: the same jobs, translated; joins, windows and **30 million rows** |
| 4 | 🔀 **pandas API on Spark**: your pandas code, unchanged, on a big dataset |
| 5 | 🧪 **pandas UDFs**: Python UDF vs pandas UDF, and `applyInPandas` (a pandas function per group, in parallel) |
| 6 | 🧩 **Final exercise**: the VIP customers of MediaMas, and the benchmark chart |

**How it works.** Same as in the previous sessions: every step has a *goal* and the *expected result*, the code cells are **empty: you write the code** (except Part 0 and the cells marked 🎨 or *ready-made*), and every question has a box below it where **you write the answer**. Run the cells **in order**.

> ⚠️ A Colab runtime is temporary: if it is recycled you lose the variables and the files. Run Part 0 again and continue from where you were.
> ⏱️ Keep the timings that your cells print: in Part 6 a chart compares them.

# Part 0 – Install Spark (ready-made)

Same as in Session 3: Java 17 + PySpark **4.3.0** (`pip`) and a `SparkSession` in **local mode** (`local[*]`: one JVM that uses all the cores of the Colab machine). This time we give the driver **4 GB** of memory and switch on **Apache Arrow**, the columnar format that Spark and pandas use to exchange data quickly.

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [ ]:
%%bash
apt-get -qq update
apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
pip uninstall -y -q dataproc-spark-connect 2>/dev/null || true
pip install -q "pyspark[pandas_on_spark]==4.3.0"
java -version 2>&1 | head -n 1

In [ ]:
import time, psutil, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("MediaMas-pandas")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())

warnings.filterwarnings("ignore")                                   # keep the output clean
pd.options.display.float_format = "{:,.2f}".format                   # no scientific notation
results = {"pandas": {}, "spark": {}, "pandas_on_spark": {}}      # seconds per number of rows (we fill it along the session)
print("pandas", pd.__version__, "| Spark", spark.version, "| cores:", spark.sparkContext.defaultParallelism,
      "| RAM: %.1f GB" % (psutil.virtual_memory().total / 1e9))

# Part 1 – 🐼 pandas: BlacFraidi on one machine

`TrainBF` is the file of Session 3: **250 000 order lines** of the campaign (date and hour, customer, city, channel, product, category, price, quantity, discount, payment, `rating`, `returned`, `delivery_days`).
pandas keeps **everything in the memory** of the machine, in a *DataFrame* made of columns (NumPy arrays). Its golden rule: **think in columns, not in rows**.

## 1.1 Load the data with pandas

Download `https://github.com/AntSanVar/DistributedSystemData/raw/refs/heads/main/Session03/TrainBF.zip` and read it into a pandas DataFrame `df`. pandas can read the **zip directly** (`pd.read_csv('TrainBF.zip', ...)`); ask it to parse `order_date` as a date (`parse_dates`). Print the **shape**, the **types** of the columns, the **memory** that `df` uses in MB (`df.memory_usage(deep=True).sum()`) and the first 5 rows.

<details><summary>💡 Hint</summary>

`!wget -q URL`; `pd.read_csv('TrainBF.zip', parse_dates=['order_date'])`; `df.shape`, `df.dtypes`, `df.head()`.

</details>

✅ **Expected:** `shape: (250000, 18)`, types such as `order_date: datetime64`, `unit_price: float64`, `rating: float64` (it has holes) and `object` for the texts. The DataFrame uses **many times the size of the zip** (see the number printed): the text columns (`object`) store one Python string per cell.

In [ ]:
# Your code here

❓ **Question:** The zip file is 5 MB. How much memory does the DataFrame use? Which kind of columns do you think weigh the most?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 1.2 Holes and statistics

Count the **missing values per column** (`isna().sum()`) and show the **statistics** (`describe()`) of `unit_price`, `quantity`, `discount_pct` and `rating`.

<details><summary>💡 Hint</summary>

`df.isna().sum()`; `df[[...]].describe().round(2)`. In a notebook only the **last** expression of a cell is displayed: use `print` or `display` for the others.

</details>

✅ **Expected:** `rating` has **87 390** nulls and `delivery_days` **77 904** (the same as in Spark; they are missing *by design*). `describe()` gives count, mean, std, min, quartiles and max: for example the mean `discount_pct` is about 20 and `rating` has a count lower than 250 000.

In [ ]:
# Your code here

❓ **Question:** Which columns have nulls and why? Compare with what you found with Spark in Session 3: is it the same?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 1.3 Vectorised vs row by row

Create the column `revenue` = `unit_price × quantity × (1 − discount_pct / 100)` rounded to 2 decimals **in a vectorised way** (operations between whole columns) and **measure the time** (`time.perf_counter()`).
Then compute the same with `df.head(25_000).apply(lambda r: ..., axis=1)` (**row by row**), measure it, and print both times, how long the `apply` would take for the 250 000 rows and the total revenue.

<details><summary>💡 Hint</summary>

Vectorised: `df['revenue'] = (df['unit_price'] * df['quantity'] * (1 - df['discount_pct'] / 100)).round(2)`. Row by row: `sample.apply(lambda r: round(r['unit_price'] * ..., 2), axis=1)`.

</details>

✅ **Expected:** The vectorised line takes a few **milliseconds**; `apply(axis=1)` needs seconds for the whole file (**hundreds or thousands of times slower**). The total revenue is about **25,525,101.36**: it differs by ~76 € from the 25,525,177.83 of Spark because pandas rounds *half to even* (and works with binary floats) while Spark rounds *half up*. A vectorised operation runs in compiled code over a whole NumPy array; `apply(axis=1)` calls a Python function 250 000 times.

In [ ]:
# Your code here

❓ **Question:** How many times slower is `apply(axis=1)`? Why? What does this tell you about how you should write pandas code? And why is the total not exactly the 25,525,177.83 of Session 3 (hint: how does each library round a 0.005?).

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 1.4 Channels and categories with `groupby`

With **named aggregation** compute, for each `channel`, the number of lines (`orders`), the units (`units`), the `revenue` and the mean discount (`discount`), rounded (0 and 1 decimals) and sorted by revenue (descending). Do the same for each `category`. Show both tables.

<details><summary>💡 Hint</summary>

`df.groupby('channel').agg(orders=('order_id', 'count'), units=('quantity', 'sum'), revenue=('revenue', 'sum'), discount=('discount_pct', 'mean'))`, then `.round({'revenue': 0, 'discount': 1}).sort_values('revenue', ascending=False)`.

</details>

✅ **Expected:** Channels: **Online** ≈ 13.2 M€ (130 156 lines), **Store** ≈ 8.0 M€ and **App** ≈ 4.4 M€. Categories: **Electronics** first (≈ 17.6 M€) and **Fashion** with the biggest discount (about 30 %). The numbers are those of Spark in Session 3 up to a few euros (see the rounding note of 1.3): same data, same logic.

In [ ]:
# Your code here

❓ **Question:** Do you get the same numbers as with Spark (compare with Session 3)? Which category has the biggest discount?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 1.5 Days and channels: `pivot_table`

Build with `pivot_table` the revenue **per day (rows) and per channel (columns)**, in million €, rounded to 2 decimals, and show it. Then run the ready-made 🎨 cell below to draw it.

<details><summary>💡 Hint</summary>

`df.pivot_table(index='order_date', columns='channel', values='revenue', aggfunc='sum')`, divided by `1e6` and `.round(2)`.

</details>

✅ **Expected:** A table of **11 rows** (20-30 November) and 3 columns (`App`, `Online`, `Store`). **2026-11-26** (the BlacFraidi Friday) is by far the biggest day: about **7.1 M€** (more than three times a normal day), half of them (**3.67 M€**) **Online**.

In [ ]:
# Your code here

❓ **Question:** Which day is the BlacFraidi? How much of the revenue of that day comes from Online?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

In [ ]:
ax = daily.plot(kind="bar", stacked=True, figsize=(10, 3.8), color=["#9467bd", "#ff7f0e", "#1f77b4"])
ax.set_ylabel("revenue (million €)"); ax.set_title("BlacFraidi: revenue per day and channel (pandas)")
ax.set_xticklabels([d.strftime("%d %b") for d in daily.index], rotation=45)
plt.tight_layout(); plt.show()

### The margins table (ready-made)

Finance sends you a small table with the **cost of every category as a percentage of the sale price**. We will use it to compute the **profit** with a `merge` (a *join*).

In [ ]:
margins = pd.DataFrame({
    "category": ["Electronics", "Home", "Fashion", "Toys", "Sports", "Beauty", "Books", "Grocery"],
    "cost_pct": [94, 55, 40, 65, 62, 35, 70, 88]})
margins

## 1.6 Profit with `merge`

Join `df` with `margins` by `category` (`merge`, `how='left'`), add `profit = revenue × (1 − cost_pct / 100)` and show, per category, the `revenue`, the `profit` (rounded to 0 decimals) and the margin in %, sorted by profit. Print also how many rows the joined table has.

<details><summary>💡 Hint</summary>

`m = df.merge(margins, on='category', how='left')`; `m.groupby('category')[['revenue', 'profit']].sum()`; margin = `100 * profit / revenue`.

</details>

✅ **Expected:** `rows after the join: 250000` (a *left* join with unique keys neither adds nor loses rows). The **order changes**: Electronics is **first in revenue** (17.6 M€) but its margin is only 6 %, so it falls to **third place in profit**, behind **Fashion** and **Home** (margins of 60 % and 45 %).

In [ ]:
# Your code here

❓ **Question:** Is the ranking by profit the same as the ranking by revenue? What would the management do with this?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

# Part 2 – 🧱 The wall: when the data grows

pandas is fantastic... **while the data fits in the memory** of one machine. How much memory do we need and how does the time grow? First, a trick to *delay* the wall; then we make the data grow.

## 2.1 A diet for the DataFrame

Make a copy `slim` of `df` and reduce its memory: convert the text columns (`object`) to **`category`** (each different text is stored once) and **downcast** the `int64` columns (`pd.to_numeric(..., downcast='integer')`) and the `float64` ones to `float32`. Print the memory **before and after** and the 3 heaviest columns of `df`.

<details><summary>💡 Hint</summary>

`for c in slim.select_dtypes(include=['object', 'string']).columns: slim[c] = slim[c].astype('category')`; the same loop for `int64` and `float64`. `df.memory_usage(deep=True).sort_values(ascending=False).head(3)`.

</details>

✅ **Expected:** A reduction of about **11 times** (130 MB → 11 MB; the exact figure depends on the pandas version). The heaviest columns are the `object` ones (`city`, `category`, `product_id`... about 14 MB each). `float32` has only ~7 significant digits: fine for a rating, **dangerous for money** (rounding errors accumulate when you add millions of values).

In [ ]:
# Your code here

❓ **Question:** How much memory did you save? What do you lose with `float32` and why would you not use it for the money?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

### 🎬 Let the data grow (ready-made)

The cell below **simulates more campaigns**: it repeats `TrainBF` *k* times, shifting the dates one year back for every copy (`order_date − 365·k` days) and giving each copy its own order ids. Spark writes the result as **Parquet** (a columnar, compressed format; later you will see why it matters) in 3 sizes: **0.25 M, 1 M and 3 M rows**. It also stores in `bytes_per_row` how much memory your pandas DataFrame needs per row.

In [ ]:
import subprocess
!unzip -o -q TrainBF.zip
base = spark.read.csv("TrainBF.csv", header=True, inferSchema=True)

def make_big(copies):
    path = f"/content/big_{copies}"
    (base.crossJoin(spark.range(copies).withColumnRenamed("id", "campaign"))
         .withColumn("order_id", F.col("order_id") + F.col("campaign") * 1000000)
         .withColumn("order_date", F.date_sub("order_date", (F.col("campaign") * 365).cast("int")))
         .drop("campaign")
         .repartition(8)
         .write.mode("overwrite").parquet(path))
    return path

bytes_per_row = df.memory_usage(deep=True).sum() / len(df)
for copies in (1, 4, 12):
    t = time.perf_counter()
    path = make_big(copies)
    size = subprocess.check_output(["du", "-sh", path]).split()[0].decode()
    print("%-14s %9d rows | %5s on disk (Parquet) | written in %.0f s" % (path, spark.read.parquet(path).count(), size, time.perf_counter() - t))
print("pandas needs about %.0f bytes of RAM per row" % bytes_per_row)

## 2.2 pandas against the clock

Write the function `pandas_job(path)` that, with **pandas**: reads the Parquet folder (`pd.read_parquet`), adds `revenue`, and groups by `category` and `channel` computing the lines (`orders`), the `revenue` and the mean of `returned` (return rate). It must return `(seconds, memory in MB of the DataFrame, result)`.
Run it for the 3 sizes (1, 4 and 12 copies = 0.25 M, 1 M and 3 M rows), store the seconds in `results['pandas'][rows]` and print a line per size.

<details><summary>💡 Hint</summary>

`d = pd.read_parquet(path)`; `time.perf_counter()` at the start and at the end; `d.memory_usage(deep=True).sum() / 1e6`; `rows = copies * 250_000`.

</details>

✅ **Expected:** Time and memory grow **roughly linearly** with the rows (×4 rows ≈ ×4 time and ×4 memory): a few seconds and about 1-2 GB for 3 M rows. For **30 M rows** pandas would need about **16 GB**, more than the free RAM: the notebook would **crash**. pandas has hit **the wall of the memory of one machine**.

In [ ]:
# Your code here

❓ **Question:** How does the time grow when you multiply the rows by 4? And the memory? Estimate the RAM needed for 30 million rows and compare it with the RAM of this machine.

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

# Part 3 – ⚡ From pandas to the PySpark DataFrame

Spark splits the data into **partitions** that it processes in parallel, and it does not need all of them in memory at the same time. The API is similar to pandas, but **lazy** and **immutable** (Session 3). A pocket dictionary:

| pandas | PySpark DataFrame |
|---|---|
| `pd.read_parquet(p)` | `spark.read.parquet(p)` |
| `df['revenue'] = a * b` | `df = df.withColumn('revenue', F.col('a') * F.col('b'))` |
| `df[df.x > 3]` | `df.filter(F.col('x') > 3)` |
| `df.groupby('k').agg(n=('id', 'count'), s=('v', 'sum'))` | `df.groupBy('k').agg(F.count('id').alias('n'), F.sum('v').alias('s'))` |
| `a.merge(b, on='k', how='left')` | `a.join(F.broadcast(b), 'k', 'left')` |
| `df.sort_values('v', ascending=False)` | `df.orderBy(F.desc('v'))` |
| `df['v'].cumsum()` | `F.sum('v').over(Window.orderBy('d').rowsBetween(...))` |
| `df.head()` / `df.describe()` | `df.show()` / `df.summary()` |
| the whole table is a *Python object in RAM* | the table is a **plan**; nothing runs until an **action** (`show`, `count`, `collect`, `toPandas`, `write`) |

## 3.1 The same job, in Spark

Write `spark_job(path)` that does **exactly what `pandas_job` did**, with the DataFrame API: `spark.read.parquet`, `withColumn('revenue', ...)`, `groupBy('category', 'channel').agg(...)` and, to force the execution, `toPandas()` of the (tiny) result. It returns `(seconds, result)`.
Run it once on `big_1` as a *warm-up* (the JVM compiles the code the first time) and then for 1, 4 and 12 copies, storing the seconds in `results['spark'][rows]`.

<details><summary>💡 Hint</summary>

`F.round(F.col('unit_price') * F.col('quantity') * (1 - F.col('discount_pct') / 100), 2)`; `F.count('*')`, `F.sum('revenue')`, `F.avg('returned')`; `out.toPandas()` is the action.

</details>

✅ **Expected:** Spark gives the **same table** as pandas, but with **similar or even worse times** for these sizes: this Colab machine has only **2 cores**, there is JVM and Python↔JVM overhead, and the data still fits in memory. Spark is not magic: it **wins when the data is huge or when you have a cluster** (the same code on 100 machines).

In [ ]:
# Your code here

❓ **Question:** Is Spark faster than pandas for these sizes? Why (or why not)? (Look at the number of cores of your machine at the start of the session.)

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 3.2 A join with a broadcast

Read `/content/big_1`, add `revenue`, and compute the profit per category as in 1.6: convert `margins` to a Spark DataFrame (`spark.createDataFrame(margins)`), **join** it with `F.broadcast(...)`, add `profit`, and show per category the `revenue` and the `profit` (rounded, as **long**), sorted by profit. Then look at the plan with `explain()`.

<details><summary>💡 Hint</summary>

`sdf.join(F.broadcast(spark.createDataFrame(margins)), 'category', 'left')`; `F.round(F.sum('revenue')).cast('long')`; `profit_s.explain()` and look for `BroadcastHashJoin`.

</details>

✅ **Expected:** The same table as in 1.6 (same order by profit). The plan contains a **`BroadcastHashJoin`**: `margins` is tiny (8 rows), so Spark copies it to every worker and the big table is **not shuffled** to do the join (a *shuffle* sends data through the network and is the most expensive thing in Spark).

In [ ]:
# Your code here

❓ **Question:** Do you get the numbers of step 1.6? Which type of join appears in the plan and why is `margins` sent (broadcast) to all the workers?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 3.3 A window: the running total

In pandas you would write `daily['revenue'].cumsum()`. In Spark, **rows have no order** unless you ask for it: you need a **window**. Compute the revenue **per day** from `sdf`, then add `cumulative` (the running total, with `Window.orderBy('order_date').rowsBetween(Window.unboundedPreceding, Window.currentRow)`) and `pct` = the cumulative as a percentage of the total (1 decimal). Show the 11 days (revenue and cumulative as long).

<details><summary>💡 Hint</summary>

`from pyspark.sql import Window`; `daily_s = sdf.groupBy('order_date').agg(F.sum('revenue').alias('revenue'))`; the total is `sdf.agg(F.sum('revenue')).first()[0]`.

</details>

✅ **Expected:** 11 rows with a growing `cumulative` that ends in **25 525 178** and `pct` = 100.0. When the 26th ends, about **58 %** of the whole campaign has already been sold in 7 days. Spark warns `No Partition Defined for Window operation`: without `partitionBy` **all the rows go to one partition** (fine for 11 rows; with 1 000 stores you would use `partitionBy('store')` to compute 1 000 totals in parallel).

In [ ]:
# Your code here

❓ **Question:** What percentage of the campaign was sold when the BlacFraidi day (26th) ended? Why does Spark warn that no partition is defined, and what would you do with 1 000 stores?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

### 🎬 30 million rows (ready-made)

Now **120 campaigns**: **30 000 000 rows** (≈ 16 GB as a pandas DataFrame, more than the RAM of this machine). Writing it takes 3-4 minutes.

In [ ]:
t = time.perf_counter()
path = make_big(120)
size = subprocess.check_output(["du", "-sh", path]).split()[0].decode()
print("%s: %d rows | %s on disk | written in %.0f s" % (path, spark.read.parquet(path).count(), size, time.perf_counter() - t))
print("pandas would need ~%.0f GB of RAM; this machine has %.1f GB" % (30e6 * bytes_per_row / 1e9, psutil.virtual_memory().total / 1e9))

## 3.4 Over the wall

Run `spark_job` on `/content/big_120` and store the seconds in `results['spark'][30_000_000]`. Print the time and the table (sorted by revenue, top 5).

<details><summary>💡 Hint</summary>

The same call as before: `secs, res = spark_job('/content/big_120')`; `res.sort_values('revenue', ascending=False).head(5)`.

</details>

✅ **Expected:** A time of **about 20-30 seconds** and a top 5 led by *Electronics* in `Online`, with revenues **120 times** those of one campaign. Spark reads the Parquet **partition by partition** (and only the columns it needs), aggregates each partition and merges the small partial results: it never holds the 30 M rows in memory, and **the same code would run on a cluster**.

In [ ]:
# Your code here

❓ **Question:** Spark finished a job that pandas cannot even start. How can it do it with only 4 GB of driver memory? (Think about partitions and what Spark keeps in memory.)

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

# Part 4 – 🔀 The pandas API on Spark

Rewriting every job in the DataFrame API is a lot of work. Since Spark 3.2 you have **`pyspark.pandas`** (the old *Koalas*): the **pandas syntax** on top of Spark DataFrames. In many cases you only change the import:

```python
import pandas as pd            ->      import pyspark.pandas as ps
df = pd.read_parquet(path)     ->      psdf = ps.read_parquet(path)
```
Behind the scenes each operation becomes a Spark plan (so it is lazy and distributed). One setting is important: pandas has a numeric **index**; Spark does not. If you ask `pyspark.pandas` for a sequential index it must number all the rows through **one single partition**. With `distributed` the ids are generated in parallel (they are unique but not consecutive).

## 4.1 Your pandas code, on 30 million rows

Set `ps.options.compute.default_index_type = 'distributed'`, and for each size (1, 4, 12 and 120 copies) read the Parquet folder with `ps.read_parquet`, add `revenue` and run **the same `groupby` of step 1.4** (by `channel`, named aggregation, sorted by revenue). Force the execution with `.to_pandas()` and store the seconds in `results['pandas_on_spark'][rows]`. Print the table of the biggest size.

<details><summary>💡 Hint</summary>

`import pyspark.pandas as ps`; `psdf = ps.read_parquet(path)`; the code of `summary` of 1.4 works almost unchanged on `psdf`; `result.to_pandas()` converts to a real pandas DataFrame (only for small results!).

</details>

✅ **Expected:** Almost **no changes**: the import, `read_parquet`, and `.to_pandas()` to get the final small table. For 30 M rows the table has the 3 channels with **Online ≈ 1 584 M€** (120 × 13.2 M€). The times are similar to those of the DataFrame API (a little higher: an extra layer of translation).

In [ ]:
# Your code here

❓ **Question:** How much of your pandas code did you have to change? Compare the time with the PySpark DataFrame version of step 3.4.

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 4.2 Moving between the three worlds

Take the Spark DataFrame `sdf` of step 3.2 and convert it to the pandas API with `sdf.pandas_api()`. Compute the **revenue per `channel` and `payment`**, sorted, and keep the first 5. Then print the **type** of: the pandas-on-Spark object, the result converted with `.to_pandas()` (a *real* pandas Series) and `psdf.to_spark()`.

<details><summary>💡 Hint</summary>

`p = sdf.pandas_api()`; `p.groupby(['channel', 'payment'])['revenue'].sum().sort_values(ascending=False).head(5)`; `type(...)`.

</details>

✅ **Expected:** The 5 best pairs (`Online / Card` first with about 7.3 M€, then `Store / Card` with 4.9 M€) and three types: `pyspark.pandas.frame`, `pandas.core.series` (**all the data now lives in the driver memory**) and `pyspark.sql.classic.dataframe`. You can move between the three APIs: pandas-on-Spark ↔ Spark DataFrame (`pandas_api()` / `to_spark()`) and → pandas (`to_pandas()`, only when the result is small).

In [ ]:
# Your code here

❓ **Question:** When would you convert to a real pandas object (`to_pandas`)? What can go wrong if you do it on the whole 30 M rows table?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 4.3 Under the hood

pandas-on-Spark is only a layer. Print the **Spark plan** of `psdf.groupby('channel')[['revenue']].sum()` with `.spark.explain()` (`psdf` is the 30 M rows DataFrame of 4.1) and the value of the option `compute.default_index_type`.

<details><summary>💡 Hint</summary>

`psdf.groupby('channel')[['revenue']].sum().spark.explain()`; `ps.get_option('compute.default_index_type')`.

</details>

✅ **Expected:** A physical plan with `FileScan parquet`, a **partial `HashAggregate`**, an `Exchange` (the shuffle) and a final `HashAggregate`: **the same Catalyst plan of Session 3**. Hard to distribute: everything that depends on the **order or position** of rows (`iloc`, `shift`, `cumsum` over the whole table, `rolling`, sequential index), because Spark has no global order of rows.

In [ ]:
# Your code here

❓ **Question:** What do you recognise in the plan? Which of the pandas functions that you know do you think are **hard** to distribute (think about rows, order and index)?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

# Part 5 – 🧪 pandas inside Spark: pandas UDFs

Sometimes you have a **function written in pandas** (or in NumPy, scikit-learn...) and you want Spark to run it in parallel. Two tools:
- **`pandas_udf`**: a function that receives a **whole column** (a `pd.Series` with thousands of values) instead of one value at a time. The data travels between the JVM and Python in **Arrow** batches.
- **`applyInPandas`**: Spark splits the DataFrame into **groups**, gives each group to your function as a **pandas DataFrame** and joins the results.

## 5.1 Python UDF vs pandas UDF vs built-in

Classify every line by the price: **`cheap`** (< 20 €), **`medium`** (< 100 €) or **`premium`**. Do it three ways on `/content/big_4` (1 M rows): a normal **Python UDF** (`@F.udf('string')`, one value at a time), a **pandas UDF** (`@F.pandas_udf('string')`, a `pd.Series` in and out, with `np.select`) and **Spark built-in functions** (`F.when(...).otherwise(...)`). For each one count the lines per band (`groupBy('band').count()`) and print the **time**.

<details><summary>💡 Hint</summary>

UDF in Python: `def band_py(price): return 'cheap' if price < 20 else ...`. pandas UDF: `pd.Series(np.select([price < 20, price < 100], ['cheap', 'medium'], 'premium'), index=price.index)`.

</details>

✅ **Expected:** The **same counts** (363 360 cheap, 466 712 medium, 169 928 premium) with the three methods, in about **8 s** (Python UDF), **3 s** (pandas UDF) and **1 s** (built-in). Order of speed: **built-in** (fastest, runs inside the JVM and Catalyst can optimise it) < **pandas UDF** (a few times slower: Arrow batches + vectorised NumPy) < **Python UDF** (slowest: every value is serialised to a Python process one by one). Rule of thumb: *built-in function if it exists; pandas UDF if you need Python/pandas/NumPy code; plain Python UDF only as a last resort*.

In [ ]:
# Your code here

❓ **Question:** Which one is the fastest and which one the slowest? Why? What is the rule of thumb to choose?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

## 5.2 A pandas function per group: peaks of sales

Write a **pandas function** `peaks(pdf)` that receives the lines of **one category**, groups them by (`order_date`, `order_hour`) summing the `revenue`, sorts by time, computes the **rolling mean of 3 hours** (`rolling(3, min_periods=1).mean()`) and marks `is_peak = revenue > mean + 2·std` of that category. It returns a DataFrame with `category`, `order_date`, `order_hour`, `revenue`, `rolling_3h`, `is_peak`.
Apply it in parallel with `sdf.groupBy('category').applyInPandas(peaks, schema)` and show the 8 peak hours with the highest revenue.

<details><summary>💡 Hint</summary>

`schema = 'category string, order_date date, order_hour long, revenue double, rolling_3h double, is_peak boolean'`; cast `order_hour` to `int64` inside the function; `pdf['category'].iloc[0]`.

</details>

✅ **Expected:** The 8 biggest peaks are all of **Electronics** (the biggest category) on **2026-11-26**, the BlacFraidi Friday: **hour 0** (the midnight opening of the offers, 483 k€ in one hour), **noon** (12 h) and the **afternoon-evening hours** (17-22 h). `applyInPandas` runs your **ordinary pandas function once per group, in parallel**; the limit is that **each group must fit in the memory** of one worker (a group is a single pandas DataFrame).

In [ ]:
# Your code here

❓ **Question:** When do the peaks of sales happen? Why is `applyInPandas` useful and what is the limit (hint: what must fit in memory)?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

# Part 6 – 🧩 Final exercise and conclusions

### 🎨 The benchmark (ready-made)

The cell draws **your** timings of the session (left) and the **RAM that pandas would need** compared with the RAM of the machine (right).

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.9))
for key, color, mark in (("pandas", "#2ca02c", "o"), ("spark", "#ff7f0e", "s"), ("pandas_on_spark", "#1f77b4", "^")):
    xs = sorted(results[key])
    if xs:
        a.plot([x / 1e6 for x in xs], [results[key][x] for x in xs], marker=mark, color=color, label=key.replace("_", " "))
a.axvline(30, color="gray", ls=":"); a.text(0.97, 0.06, "pandas cannot load 30 M rows", transform=a.transAxes, color="gray", ha="right")
a.set_xscale("log"); a.set_yscale("log"); a.set_xlabel("million rows"); a.set_ylabel("seconds"); a.set_title("Time of the same job"); a.legend()

rows = np.array([0.25, 1, 3, 12, 30])
gb = rows * 1e6 * bytes_per_row / 1e9
b.bar([f"{r:g}" for r in rows], gb, color=["#2ca02c" if g < psutil.virtual_memory().total / 1e9 else "#d62728" for g in gb])
b.axhline(psutil.virtual_memory().total / 1e9, color="k", ls="--"); b.text(0, psutil.virtual_memory().total / 1e9 * 1.03, "RAM of this machine")
b.set_xlabel("million rows"); b.set_ylabel("GB needed by pandas"); b.set_title("Memory of a pandas DataFrame")
plt.tight_layout(); plt.show()

❓ **Question:** Looking at the left chart: for which sizes is pandas the best choice? When does Spark start to make sense? Add your own conclusion in two lines.

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

### The VIP customers of MediaMas

The marketing department wants the **VIP customers**: those whose **total spend over the 12 campaigns** is in the **top 2 %**. The table is `big_12` (3 M rows). Use **the pandas API on Spark** (you already know the pandas code!).

## 6.1 The VIP list

Read `/content/big_12` with `ps.read_parquet`, add `revenue`, and group **by `customer_id`**: `orders` (lines), `spend` (sum of revenue) and `returned` (mean of `returned`). Find the **98th percentile** of `spend` (`quantile(0.98)`), keep the customers at or above it, sorted by `spend` (descending). Print the number of customers, the number of VIPs and the first 5 VIPs (as a real pandas table, rounded). Save the VIPs with `to_parquet('/content/vip_customers')`.

<details><summary>💡 Hint</summary>

`cust = psc.groupby('customer_id').agg(orders=('order_id', 'count'), spend=('revenue', 'sum'), returned=('returned', 'mean'))`; `limit = cust['spend'].quantile(0.98)`; `vip = cust[cust['spend'] >= limit].sort_values('spend', ascending=False)`.

</details>

✅ **Expected:** `customers: 88274 | VIPs: 1774` (2 %), a limit of about **19 800 €**, a mean spend of **3 470 €** over the 12 campaigns and the best customers above **108 000 €**: more than **30 times the mean**. The result is saved as a Parquet folder (`/content/vip_customers`), ready for the marketing team.

In [ ]:
# Your code here

❓ **Question:** How many VIPs are there and how much do the best ones spend compared with the average customer?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

### Your result

Paste the 5 rows of step 6.1 in the box below.

✍️ **Your result** *(double-click, then paste the 5 rows below, one per line inside this quote)*:

> 

# Part 7 – Clean up

## 7.1 Stop Spark and free the disk

Stop the Spark session and delete the big Parquet folders (`/content/big_*`) with `shutil.rmtree`. Print the free disk space before and after (`shutil.disk_usage('/content').free`).

<details><summary>💡 Hint</summary>

`spark.stop()`; `glob.glob('/content/big_*')`; `shutil.rmtree(p, ignore_errors=True)`.

</details>

✅ **Expected:** The free space **grows** by the size of the 4 Parquet datasets (about 0.6 GB). `spark.stop()` closes the JVM and frees its memory and cores for the next notebook.

In [ ]:
# Your code here

❓ **Question:** Why is it good practice to stop the session and delete temporary data?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

### Final questions

Answer each one in the box below it.

❓ **Question:** Why is pandas faster than Spark for 250 000 rows, and why does it stop working for 30 million? What does Spark change in the way the data is stored and processed?

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

❓ **Question:** You have a team that writes pandas code. Compare the three ways of scaling it (PySpark DataFrame API, pandas API on Spark, pandas UDF / `applyInPandas`): advantages, risks and when you would choose each.

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 

❓ **Question:** Write **three sentences for the CFO of MediaMas**: what the VIP analysis tells us, how long the 30 M row analysis took, and why next year, with 10 times more data, the process will keep working.

✍️ **Your answer** *(double-click this cell and write below this line)*:

> 